# HumanLM VM Forward Pass

Notebook ini **khusus untuk VM** yang punya RAM cukup.

Jangan jalankan notebook ini di laptop 16 GB jika model masih BF16 penuh.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "snap-stanford/humanlm-opinion"
DEVICE = "cpu"
TORCH_DTYPE = torch.bfloat16

print(MODEL_NAME, DEVICE, TORCH_DTYPE)

## 1. Load Tokenizer dan Model

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=TORCH_DTYPE,
    device_map=DEVICE,
)
model.eval()

print(type(model).__name__)
print("parameter_count:", sum(p.numel() for p in model.parameters()))

## 2. Forward Pass Kecil

In [ ]:
messages = [
    {"role": "user", "content": "Who are you?"},
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
)

inputs = {k: v.to(model.device) for k, v in inputs.items()}
print({k: tuple(v.shape) for k, v in inputs.items()})

In [ ]:
with torch.no_grad():
    outputs = model(**inputs, output_hidden_states=True, return_dict=True)

print("logits shape:", tuple(outputs.logits.shape))
print("num hidden_states:", len(outputs.hidden_states))
print("embedding output shape:", tuple(outputs.hidden_states[0].shape))
print("last hidden state shape:", tuple(outputs.hidden_states[-1].shape))

## 3. Inspect Layer Vectors

In [ ]:
hidden_state_shapes = [tuple(h.shape) for h in outputs.hidden_states]
hidden_state_shapes[:10], hidden_state_shapes[-3:]

In [ ]:
last_token_vectors = [h[0, -1, :].float() for h in outputs.hidden_states]
print("num layer vectors:", len(last_token_vectors))
print("vector dim:", last_token_vectors[-1].shape[0])

## 4. Generate Pendek

In [ ]:
with torch.no_grad():
    generated = model.generate(**inputs, max_new_tokens=40, do_sample=False)

new_tokens = generated[0][inputs['input_ids'].shape[-1]:]
print(tokenizer.decode(new_tokens, skip_special_tokens=True))